# Bitcoin Regime-Conditional Robustness and Temporal Stability

## Role
This notebook evaluates whether the frozen Bitcoin forecast vectors remain reliable when market conditions or evaluation periods change, rather than only looking good on aggregate (Notebook 07).

## Inputs
The authoritative frozen Bitcoin forecast matrix (`results/validated_forecasts.csv`), the canonical training target/return series, the training-defined threshold artifact (`results/bitcoin_regime_thresholds_training.csv`), the regime-conditional metrics artifact (`results/bitcoin_regime_robustness_training_defined.csv`), and the temporal-stability artifact (`results/bitcoin_temporal_stability.csv`).

## Outputs
Training-defined regime thresholds, regime sample-size evidence, regime-conditional performance, temporal-segment performance, a regime robustness ranking, and a temporal-stability ranking.

## Depends On
`07_Bitcoin_Forecast_Freeze_and_Validation.ipynb`
`08_Bitcoin_Naive_Audit.ipynb`

## Authoritative Status
`AUTHORITATIVE CONDITIONAL PERFORMANCE ANALYSIS`

## What This Notebook Does Not Do
No model training. No forecast regeneration. No uncertainty calibration. No statistical significance testing. No Trust Score synthesis. No comprehensive adversarial robustness analysis.

# 1. Objective and Research Questions

### Q1 — Regime-Conditional Robustness

> Do model errors remain controlled across different volatility and return-movement regimes defined from training data only?

### Q2 — Temporal Stability

> Does model performance remain reasonably stable across Earlier, Middle, and Later portions of the frozen test period, or is aggregate performance driven disproportionately by one temporal segment?

> All regime thresholds are estimated from training data only and are then applied unchanged to the final test period. Final-test outcomes do not determine the regime thresholds.

# 2. Setup

Standard project-root discovery. Only the frozen forecast matrix, the canonical train/test target, the training-derived threshold and derived-metric artifacts, and the repository's canonical metric helpers are loaded. No forecasting model is fit and no final forecast is recreated.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.bitcoin_pipeline import *

RUN_GENERATION = False
PROMOTE_TO_AUTHORITATIVE = False

# 3. Data and Method

## 3.1 Scope and Leakage Guardrail

> These analyses are conditional diagnostics, not comprehensive claims of robustness or universal generalisation.

This project's preferred public terminology is **Regime-Conditional Robustness** and **Temporal Stability**; "Generalisation" is avoided here except where legacy artifact column names already use it (`bitcoin_regime_robustness_training_defined.csv`'s companion artifacts use "generalisation" historically -- the underlying evidence is unchanged, only the notebook's own language is standardised).

Regime thresholds are training-defined (Section 3.2) and applied unchanged to the frozen test observations; every model is evaluated on exactly the same regime masks and the same temporal segments, so no regime or segment is tailored to any specific model.

## 3.2 Regime Definition Method

Verified directly from `src.bitcoin_pipeline.training_regime_thresholds` / `test_regime_masks`: the volatility measure is a **14-day rolling standard deviation of daily percentage returns** (`pct_change().rolling(14, min_periods=7).std()`); the movement measure is the **daily percentage return itself**. Thresholds are training-period quantiles of these two training-period series, then applied unchanged to test-period values of the same series.

In [ ]:
regime_method = pd.DataFrame([
    {'Regime': 'Low Volatility', 'Training Variable': '14-day rolling std of daily % return', 'Training Rule': 'value <= training 33rd percentile', 'Interpretation': 'Calmer-than-usual short-horizon price movement'},
    {'Regime': 'High Volatility', 'Training Variable': '14-day rolling std of daily % return', 'Training Rule': 'value >= training 67th percentile', 'Interpretation': 'More turbulent-than-usual short-horizon price movement'},
    {'Regime': 'Major Upward Move', 'Training Variable': 'Daily % return', 'Training Rule': 'value >= training 80th percentile', 'Interpretation': 'Unusually large single-day price increase'},
    {'Regime': 'Major Downward Move', 'Training Variable': 'Daily % return', 'Training Rule': 'value <= training 20th percentile', 'Interpretation': 'Unusually large single-day price decrease'},
]).set_index('Regime')
regime_method

> Defining thresholds from the test set would condition the evaluation design on the outcomes being evaluated and would weaken the independence of the robustness analysis.

## 3.3 Temporal Segmentation Method

> The 1,061 final-test observations are divided chronologically into approximately equal Earlier, Middle, and Later segments, preserving order and avoiding overlap.

Verified from `src.bitcoin_pipeline.temporal_segments`: the chronological test index is split with `numpy.array_split(index, 3)`, which produces contiguous, non-overlapping, order-preserving thirds without requiring the count to divide evenly. Thirds are used because the split is simple, interpretable, model-independent, leaves enough observations per segment for a stable RMSE estimate, and is intended as a temporal-stability diagnostic rather than a formal structural-break analysis -- the three segments are not claimed to correspond to any known market regime or economic event.

# 4. Regime-Conditional Robustness Results

## 4.1 Training-Defined Thresholds

> The thresholds below are estimated once from the training period and then held fixed during final-test evaluation.

In [ ]:
# Load the canonical split and the frozen forecast matrix.
_, target = load_bitcoin_target(ROOT)
train, test = canonical_split(target)
validated = load_validated_forecasts(ROOT)

thresholds = pd.read_csv(ROOT / 'results' / 'bitcoin_regime_thresholds_training.csv')
assert thresholds.Source.eq('Training data only').all()
thresholds

## 4.2 Regime Sample Sizes

> Conditional metrics are only as reliable as the number of observations supporting them. Regimes with substantially fewer test observations should be interpreted with greater uncertainty.

In [ ]:
robust = pd.read_csv(ROOT / 'results' / 'bitcoin_regime_robustness_training_defined.csv')
assert robust.Model.nunique() == 10

regime_sizes = robust.groupby('Regime').N.first().to_frame('N')
regime_sizes['Share of Final Test (%)'] = 100 * regime_sizes['N'] / len(test)
regime_sizes.round(2)

High Volatility is estimated from only 81 test observations (7.6% of the final test period), versus 676 for Low Volatility (63.7%) -- an 8.3x imbalance. High-volatility performance comparisons in Section 4.3-4.4 therefore carry substantially more sampling uncertainty than low-volatility comparisons, and this is not corrected for retroactively anywhere below.

## 4.3 Regime RMSE Comparison

The chart below shows RMSE for all 10 final analytical models under each of the four training-defined regimes, using identical regime masks and the identical metric for every model.

In [ ]:
regime_rmse = robust.pivot(index='Model', columns='Regime', values='RMSE')
ordered_regimes = ['Low Volatility', 'High Volatility', 'Major Upward Movement', 'Major Downward Movement']
ax = regime_rmse[ordered_regimes].plot(kind='bar', figsize=(14, 6), width=0.85)
ax.set(title='RMSE by Training-Defined Bitcoin Regime', xlabel='Model', ylabel='RMSE (USD)')
ax.tick_params(axis='x', rotation=25)
ax.grid(axis='y', alpha=0.25)
ax.legend(title='Regime', frameon=False, ncol=2)
ax.figure.tight_layout()
plt.show()

Look for two things in the grouped bars: how tall the Low Volatility bar is relative to the other three for each model (a large gap indicates regime sensitivity), and whether any model's bar ordering across regimes departs from the rest -- most models track each other's regime-to-regime shape, with Prophet and the PE-Transformer visibly the tallest across every regime.

## 4.4 Regime Robustness Ranking

`RMSE Spread = max(RMSE_regime) - min(RMSE_regime)`; `Relative RMSE Spread = RMSE Spread / mean(RMSE_regime)`. Ranked by Relative RMSE Spread, lowest first: small relative spread means more stable conditional performance across regimes, large relative spread means greater regime sensitivity. This is a stability-across-regimes measure, not a universal robustness score -- a model can rank well here while still having poor absolute accuracy in every regime (see interpretation below the table).

In [ ]:
regime_summary = pd.DataFrame({
    'Best Regime RMSE': regime_rmse.min(axis=1),
    'Worst Regime RMSE': regime_rmse.max(axis=1),
})
regime_summary['RMSE Spread'] = regime_summary['Worst Regime RMSE'] - regime_summary['Best Regime RMSE']
regime_summary['Relative RMSE Spread'] = regime_summary['RMSE Spread'] / regime_rmse.mean(axis=1)
regime_summary = regime_summary.sort_values('Relative RMSE Spread')
regime_summary.insert(0, 'Rank', range(1, len(regime_summary) + 1))
regime_summary.round(3)

Reading this table requires care: Prophet ranks 1st (smallest relative spread) not because it is robust, but because its RMSE is uniformly very high across every regime (its own absolute spread, ~4,417, is in fact the largest of the ten models) -- a large, consistently bad mean pulls the relative-spread ratio down. The 7-Day Moving Average shows the same pattern at rank 2. Among the models with genuinely competitive absolute accuracy, Chronos-Bolt-Tiny (rank 3) and TimesFM (rank 4) show the smallest relative regime sensitivity.

# 5. Temporal Stability Results

## 5.1 Balanced Segment Definition

Temporal stability is assessed by evaluating the same frozen forecast vectors separately over three chronological portions of the final test period.

In [ ]:
stability = pd.read_csv(ROOT / 'results' / 'bitcoin_temporal_stability.csv')
assert stability.Model.nunique() == 10

segment_definition = stability.groupby('Segment').agg(Start=('Start', 'first'), End=('End', 'first'), N=('N', 'first')).loc[['Earlier', 'Middle', 'Later']]
segment_definition['Share (%)'] = 100 * segment_definition['N'] / len(test)
segment_definition.round(2)

354 + 354 + 353 = 1,061, confirming the three segments are approximately balanced (within one observation of an exact third) and sum to the full final test period with no overlap.

## 5.2 Segment RMSE Comparison

RMSE for all 10 models across the Earlier, Middle, and Later segments, using identical segment boundaries for every model.

In [ ]:
segment_rmse = stability.pivot(index='Model', columns='Segment', values='RMSE')
ax = segment_rmse[['Earlier', 'Middle', 'Later']].plot(kind='bar', figsize=(14, 6), width=0.8)
ax.set(title='Bitcoin RMSE Across Earlier, Middle, and Later Test Segments', xlabel='Model', ylabel='RMSE (USD)')
ax.tick_params(axis='x', rotation=25)
ax.grid(axis='y', alpha=0.25)
ax.legend(title='Segment', frameon=False)
ax.figure.tight_layout()
plt.show()

Every model's Earlier bar is visibly the shortest of its three, and Middle and Later are broadly comparable to each other for most models -- consistent with the Section 5.3 finding that the Earlier segment is the easiest across the board, not with any single model behaving unusually in one segment.

## 5.3 Temporal Stability Ranking

Same formulas as Section 4.4, applied across the three temporal segments: `RMSE Spread = max(RMSE_segment) - min(RMSE_segment)`; `Relative Spread = RMSE Spread / mean(RMSE_segment)`. Ranked by Relative Spread, lowest (most temporally stable) first.

In [ ]:
segment_summary = segment_rmse[['Earlier', 'Middle', 'Later']].copy()
segment_summary['RMSE Spread'] = segment_rmse[['Earlier', 'Middle', 'Later']].max(axis=1) - segment_rmse[['Earlier', 'Middle', 'Later']].min(axis=1)
segment_summary['Relative Spread'] = segment_summary['RMSE Spread'] / segment_rmse[['Earlier', 'Middle', 'Later']].mean(axis=1)
segment_summary = segment_summary.sort_values('Relative Spread')
segment_summary.insert(0, 'Rank', range(1, len(segment_summary) + 1))
segment_summary.round(3)

# 6. Aggregate Accuracy vs Conditional Behaviour

The canonical full 10-model ranking is recomputed here from the same frozen matrix and the same `metric_table` function Notebook 07 uses -- not re-typed from memory -- so the comparison below is against a live, verifiable source.

In [ ]:
forecasts = forecast_series(validated, target)
aggregate_ranking = metric_table(validated.Actual, forecasts, train).sort_values('RMSE')
aggregate_rank = pd.Series(range(1, len(aggregate_ranking) + 1), index=aggregate_ranking.index, name='Aggregate RMSE Rank')

comparison = pd.DataFrame({
    'Aggregate RMSE Rank': aggregate_rank,
    'Regime Robustness Rank': regime_summary['Rank'],
    'Temporal Stability Rank': segment_summary['Rank'],
}).sort_values('Aggregate RMSE Rank')
comparison

The rankings diverge materially rather than matching. Naive leads aggregate accuracy (rank 1) but falls to rank 8 on regime robustness and rank 5 on temporal stability. Chronos-Bolt-Tiny is mediocre in aggregate (rank 7) but is the single most temporally stable model (rank 1) and the third-most regime-robust. TimesFM (aggregate rank 6) is reasonably regime-robust (rank 4) but among the least temporally stable (rank 8). The Persistence-Enhanced Log-Return Transformer is consistently weak across all three dimensions (ranks 8, 10, 10). Prophet and the 7-Day Moving Average rank well on both conditional-spread measures only because their errors are uniformly large rather than genuinely stable in an absolute sense (Section 4.4).

> Aggregate point accuracy and conditional reliability are not interchangeable dimensions.

No Trust Score or composite synthesis is calculated here -- that belongs to `12_Bitcoin_Trustworthiness_Synthesis.ipynb`.

# 7. Key Findings

### Regime-Conditional Robustness

- Excluding the two uniformly-weak models (Prophet, 7-Day Moving Average, Section 4.4), Chronos-Bolt-Tiny and TimesFM show the smallest relative RMSE spread across regimes among models with competitive absolute accuracy.
- The Persistence-Enhanced Log-Return Transformer is the most regime-sensitive model with genuinely competitive aggregate accuracy (relative spread 1.085, more than double the next-highest among the accuracy-competitive models).
- Low Volatility is the easiest regime overall (mean RMSE ~2,593 across all 10 models); Major Upward Movement and High Volatility are the hardest (mean RMSE ~4,340 and ~4,293 respectively), with Major Downward Movement intermediate (~3,871).
- Volatility and directional-move regimes are not mutually exclusive: 17 test observations are both High Volatility and Major Upward Movement, and 20 are both High Volatility and Major Downward Movement (Section 8).

### Temporal Stability

- Chronos-Bolt-Tiny is the most temporally stable model overall (relative spread 0.363); the Persistence-Enhanced Log-Return Transformer is the least stable among competitive models (0.450), with Prophet's absolute spread far larger in dollar terms despite a similar relative ranking.
- The Earlier segment is consistently the easiest across every model (lowest RMSE in all 10 cases); Middle and Later are both harder than Earlier and broadly comparable to each other, so no single model is unusually exposed to one specific later sub-period relative to the others.

### Accuracy vs Stability

Notebook 07's aggregate ranking does **not** agree with either conditional ranking. Naive's rank-1 aggregate result does not carry over to regime robustness (rank 8) or temporal stability (rank 5); Chronos-Bolt-Tiny is the clearest example of the opposite pattern, ranking mid-table in aggregate accuracy while leading temporal stability. This divergence is one of the more important trustworthiness findings from the Bitcoin case: a model should not be assumed dependable in every operating condition merely because it wins on average error, and a model that trails on average error is not automatically fragile. Models are described here only as "more/less regime-robust" or "more/less temporally stable" under this specific protocol -- not as "trustworthy," which is a broader synthesis owned by Notebook 12.

# 8. Limitations

### Regime Sample Imbalance
High Volatility (N=81) has roughly 8x fewer test observations than Low Volatility (N=676, Section 4.2). Conditional RMSE estimates for small regimes have greater sampling uncertainty than the chart in Section 4.3 visually implies.

### Threshold Choice
The 33rd/67th-percentile volatility split and the 80th/20th-percentile return-movement split are transparent and training-defined, but the specific percentile cutpoints are researcher-selected. Alternative threshold definitions were not comprehensively sensitivity-tested in this repository.

### Overlapping Regimes
The four regimes are not all mutually exclusive: Low Volatility and High Volatility cannot co-occur by construction, and Major Upward/Downward Movement cannot co-occur by construction, but volatility and directional-movement regimes can and do overlap (Section 7). A single test day can therefore contribute to more than one regime's metrics.

### Temporal Thirds
Earlier/Middle/Later thirds are diagnostic partitions for readability and balance, not statistically estimated structural-break points, and are not claimed to correspond to any known market regime or event.

### No Formal Uncertainty on Conditional Ranks
No confidence intervals or bootstrap resampling were computed for the regime or temporal-stability rankings in Section 4.4/5.3/6; rank differences of one or two positions should not be over-interpreted as definitive.

### Frozen Case Study
Findings apply to this Bitcoin asset, this frozen test period, and the past-only forecasting protocols documented in Notebooks 02-06.

# 9. Next Notebook

Next: `10_Bitcoin_Uncertainty.ipynb`

Notebook 10 moves from realised point-error behaviour to predictive uncertainty evidence: interval construction, calibration, coverage, width, and evidence availability.